# 📑 PageIndex — Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking
**By [@krishnaik06](https://youtube.com/@krishnaik06) | [krishnaik.in/liveclasses](https://krishnaik.in/liveclasses)**

---

## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.


---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + OpenAI SDK (pointed at Groq)
- Load API keys from `.env`
- Initialize both clients

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **Groq API key** from: https://console.groq.com/keys


In [ ]:
# Install required packages
%pip install -U pageindex openai python-dotenv

In [1]:
# ── Create a .env file (run this once) ──────────────────────────────────────
# A .env file lives next to this notebook. Put your real keys there:
#   PAGEINDEX_API_KEY=...   (https://dash.pageindex.ai/api-keys)
#   GROQ_API_KEY=...        (https://console.groq.com/keys)
import os
if not os.path.exists(".env"):
    with open(".env", "w") as f:
        f.write("PAGEINDEX_API_KEY=your_pageindex_key_here\nGROQ_API_KEY=your_groq_key_here\n")
    print("📝 .env template created — open it, paste your keys, then re-run the next cell")
else:
    print("✅ .env already exists")

✅ .env already exists


In [2]:
import os, json, time
from dotenv import load_dotenv

load_dotenv(override=True)

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

def _ok(k):
    return bool(k) and not k.startswith("your_")

print("PageIndex key loaded:", "✅" if _ok(PAGEINDEX_API_KEY) else "❌ Missing! Edit .env")
print("Groq key loaded:     ", "✅" if _ok(GROQ_API_KEY)      else "❌ Missing! Edit .env")
assert _ok(PAGEINDEX_API_KEY) and _ok(GROQ_API_KEY), "Fill in your keys in the .env file, then re-run this cell."

PageIndex key loaded: ✅
Groq key loaded:      ✅


In [3]:
from pageindex import PageIndexClient
from openai import OpenAI

# Groq exposes an OpenAI-compatible API, so we reuse the OpenAI SDK
# and just point it at Groq's endpoint.
LLM_MODEL = "openai/gpt-oss-120b"   # other Groq options: "openai/gpt-oss-20b", "qwen/qwen3.8-27b"

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    max_retries=6,                  # Groq free tier has tight per-minute limits; retry on 429
)

print("✅ PageIndex client ready")
print(f"✅ Groq client ready (model: {LLM_MODEL})")

✅ PageIndex client ready
✅ Groq client ready (model: openai/gpt-oss-120b)


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [4]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./iso27001.pdf"   # ISO/IEC 27001:2022 standard (26 pages)

# Already uploaded before? Paste the doc_id here to skip re-uploading
EXISTING_DOC_ID = "pi-cmwbofyjr003j0cnvpq38yxx9"                 # e.g. "pi-xxxxxxxxxxxxxxxx"

if EXISTING_DOC_ID:
    doc_id = EXISTING_DOC_ID
    print(f"♻️ Reusing existing document: {doc_id}")
else:
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"PDF not found: {os.path.abspath(PDF_PATH)} — put your PDF there or change PDF_PATH")
    print(f"📤 Uploading: {PDF_PATH}")
    result = pi_client.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"✅ Uploaded!")
    print(f"📋 Document ID: {doc_id}")
    print("   (Save this ID in EXISTING_DOC_ID to avoid re-uploading)")

♻️ Reusing existing document: pi-cmwbofyjr003j0cnvpq38yxx9


In [5]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        raise RuntimeError(f"❌ Processing failed. Check your PDF format. Details: {status_result}")
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)


   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [6]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Information security, cybersecurity",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 26,
  "summary": "This section covers the ISO/IEC 27001:2022 standard for Information Security Management Systems (ISMS), including title and copyright details, the table of contents, and the foreword outlining standard development and revision history. It details the ISMS introduction\u2014covering general principles, scope, and core requirements across clauses 4 through 10 (organizational context, leadership, planning, support, operation, performance evaluation, and improvement)\u2014as well as Annex A, which provides normative information security controls categorized into organizational, people, physical, and technological areas, and concludes with a bibliography and document metadata.",
  "text": "",
  "nodes": [
    {
      "title": "Information security, cybersecurity (intro)",
      "node_id": "0001",
      "start_index": 

In [7]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = (f"{node['start_index']}-{node['end_index']}" if node.get("start_index") else node.get("page_index", "?"))
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Information security, cybersecurity  (p.1-26)
  └─ [0001] Information security, cybersecurity (intro)  (p.1-3)
  └─ [0002] Contents  (p.3-4)
  └─ [0003] Foreword  (p.4-5)
  └─ [0004] Introduction  (p.5-17)
    └─ [0005] 0.1 General  (p.5-5)
    └─ [0006] 0.2 Compatibility with other management system standards  (p.5-7)
    └─ [0007] 1 Scope  (p.7-7)
    └─ [0008] 2 Normative references  (p.7-7)
    └─ [0009] 3 Terms and definitions  (p.7-7)
    └─ [0010] 4 Context of the organization  (p.7-8)
    └─ [0011] 5 Leadership  (p.8-9)
    └─ [0012] 6 Planning  (p.9-12)
      └─ [0013] 6.1 Actions to address risks and opportunities  (p.9-11)
      └─ [0014] 6.2 Information security objectives and planning to achieve them  (p.11-12)
    └─ [0015] 7 Support  (p.12-13)
    └─ [0016] 8 Operation  (p.13-14)
    └─ [0017] 9 Performance evaluation  (p.14-16)
    └─ [0018] 10 Improvement  (p.16-17)
  └─ [0019] Annex A (normative)  (p.17-25)
  └─ [0020] Bibliography  

In [8]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 21
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [9]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = None) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "pages":   (f"{n['start_index']}-{n['end_index']}" if n.get("start_index") else n.get("page_index", "?")),
                "summary": (n.get("summary") or n.get("text") or "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [10]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What does ISO 27001 require for information security risk assessment?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What does ISO 27001 require for information security risk assessment?



🧠 LLM Reasoning:
The query asks what ISO 27001 requires for information security risk assessment. Risk assessment is addressed in the planning phase of the ISMS, specifically in the clause about actions to address risks and opportunities. The node titled '6.1 Actions to address risks and opportunities' (node_id 0013) is the most direct source. The broader planning section (node_id 0012) also discusses the overall risk assessment process, and the context of the organization (node_id 0010) includes understanding internal and external issues that feed into risk assessment. Therefore, the most relevant nodes are 0013, 0012, and 0010.

🎯 Selected Node IDs: ['0013', '0012', '0010']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [11]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [12]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = None) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    # (capped so we stay inside Groq's per-request token limits)
    MAX_CHARS_PER_NODE = 6000
    nodes = nodes[:6]
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Pages {(f"{node['start_index']}-{node['end_index']}" if node.get("start_index") else node.get("page_index", "?"))}]\n"
            f"{(node.get('text') or node.get('summary') or 'Content not available.')[:MAX_CHARS_PER_NODE]}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = openai_client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [13]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [14]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are top management's leadership responsibilities for the ISMS?",
    tree=pageindex_tree
)

🔍 Query: What are top management's leadership responsibilities for the ISMS?



🧠 Reasoning: The query asks for top management's leadership responsibilities for the ISMS. In ISO/IEC 27001:2022, these responsibilities are detailed in Clause 5 titled 'Leadership'. The document tree shows node 0...
🎯 Retrieved node IDs: ['0011']
📄 Sections found: ['5 Leadership']



📝 Answer:
Top management’s leadership responsibilities for the ISMS are set out in **Section 5 Leadership (pages 8‑9)** and include the following actions:

* **Establishing the policy and objectives** and aligning them with the organization’s strategic direction (5.1 a).  
* **Integrating ISMS requirements** into the organization’s processes (5.1 b).  
* **Providing the resources** needed for the ISMS (5.1 c).  
* **Communicating the importance** of effective information‑security management and compliance with ISMS requirements (5.1 d).  
* **Ensuring the ISMS achieves its intended outcomes** (5.1 e).  
* **Directing and supporting personnel** to contribute to ISMS effectiveness (5.1 f).  
* **Promoting continual improvement** of the ISMS (5.1 g).  
* **Supporting other relevant management roles** to demonstrate leadership within their areas of responsibility (5.1 h).  

These items collectively define top management’s leadership duties for the information‑security management system. 

In [15]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What must be included in the information security policy?",
    "What are the required inputs to a management review?",
    "How should an organization handle a nonconformity?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)

Q: What must be included in the information security policy?
A: The information‑security policy must:

* be appropriate to the organization’s purpose (a)【5 Leadership, p.8】  
* contain information‑security objectives or a framework for setting them (b)【5 Leadership, p.8】  
* include a commitment to meet all applicable information‑security requirements (c)【5 Lead...
-------------------------------------------------------



Q: What are the required inputs to a management review?
A: **Required inputs to a management review (ISO / IEC 27001:2022 – 9.3 Management review, p. 15):**

1. **Status of actions from previous management reviews** (a)  
2. **Changes in external and internal issues** relevant to the ISMS (b)  
3. **Changes in needs and expectations of interested parties** ...
-------------------------------------------------------



Q: How should an organization handle a nonconformity?
A: An organization must manage a non‑conformity through a defined sequence of actions:

1. **React to the nonconformity** – control and correct it and deal with any consequences. (10 Improvement, p.16)  
2. **Assess whether further action is needed to eliminate the cause** – review the nonconformity, d...
-------------------------------------------------------


---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [16]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.


In [17]:

FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [18]:
# ── Expert Routing Rules — ISO/IEC 27001:2022 ───────────────────────────────
# Encodes a lead auditor's knowledge of where things live in the standard.
ISO_EXPERT_RULES = """
Route queries to the correct clause of ISO/IEC 27001:2022 using these rules:

Clause 4  Context of the organization → internal/external issues, interested parties,
                                         ISMS scope, establishing the ISMS
Clause 5  Leadership                  → top management commitment, information security
                                         policy, roles, responsibilities and authorities
Clause 6  Planning                    → risks & opportunities, risk assessment criteria (6.1.2),
                                         risk treatment, Statement of Applicability (6.1.3),
                                         security objectives (6.2), planning of changes (6.3)
Clause 7  Support                     → resources, competence, awareness, communication,
                                         documented information (create/update/control)
Clause 8  Operation                   → operational planning & control, performing risk
                                         assessments (8.2) and risk treatment (8.3)
Clause 9  Performance evaluation      → monitoring & measurement, internal audit (9.2),
                                         management review inputs/results (9.3)
Clause 10 Improvement                 → continual improvement, nonconformity & corrective action
Annex A   Controls reference          → the 93 controls: A.5 organizational, A.6 people,
                                         A.7 physical, A.8 technological

Cross-cutting rules:
- "risk" questions                    → 6.1.2 / 6.1.3 (requirements) + 8.2 / 8.3 (execution)
- "Statement of Applicability / SoA"  → 6.1.3 + Annex A
- "specific control" (access control, cryptography, backup, logging, supplier,
   incident, physical security, etc.) → Annex A
- "audit / certification readiness"   → 9.2 internal audit + 9.3 management review
- "documents / records required"      → 7.5 + the clause that mandates the record
- "what changed / edition / 2013"     → Foreword
"""
print("✅ ISO 27001 expert rules defined")

✅ ISO 27001 expert rules defined


In [19]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = None
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "pages": (f"{n['start_index']}-{n['end_index']}" if n.get("start_index") else n.get("page_index", "?")),
                     "summary": (n.get("summary") or n.get("text") or "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [20]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Which controls cover access control and identity management?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, ISO_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Which controls cover access control and identity management?

── Without Expert Rules ──


Nodes: ['0019']

── With Expert Rules ──


Nodes: ['0019']
Reasoning: The query asks for the specific controls that address access control and identity management. According to the expert routing rules, any question about a specific control (e.g., access control, identity management) should be routed to Annex A, which lists the 93 controls. In the document tree, Annex


In [21]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="What documented information is mandatory for the risk treatment process?",
    tree=pageindex_tree,
    rules=ISO_EXPERT_RULES
)
print(answer)

The organization must keep **documented information** for the risk‑treatment process that includes:

* **A Statement of Applicability** – listing the required controls, the justification for each control, whether each control is implemented, and the justification for any excluded Annex A controls.  
* **An information‑security risk‑treatment plan**.  
* **Evidence that the risk owners have approved the risk‑treatment plan and have accepted the residual risks**.  

The standard also requires that the organization **retain documented information about the risk‑treatment process itself**.  

(Section “6.1.3 Information security risk treatment”, p. 4)


---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [22]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "Summarize the main requirements of this standard."

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
The structure summaries cover the main points, so I'll answer from them.

# ISO/IEC 27001:2022 – Main Requirements

**Purpose and scope (Clause 1):** The standard sets requirements for establishing, implementing, maintaining and continually improving an Information Security Management System (ISMS). The ISMS protects the confidentiality, integrity and availability of information through risk management. The requirements are generic and apply to any organization. Excluding any requirement in Clauses 4–10 is not acceptable if the organization claims conformity.

**Core clauses (4–10)**
- **4 Context of the organization:** Identify internal and external issues. Identify interested parties and their requirements. Define the ISMS scope. Establish and maintain the ISMS.
- **5 Leadership:** Top management must show commitment and leadership. It must set and communicate an information security policy. It must assign roles, responsibilities and authorities.
- **6 Planning**
 

In [23]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What does clause 9.2 say about internal audits?",
    "How does that relate to the management review in 9.3?",
    "What outputs must the management review produce?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What does clause 9.2 say about internal audits?


🤖 Assistant: Clause 9.2 (Internal audit) is on pages 14–15. It has two parts.

**9.2.1 General**
The organization must conduct internal audits at planned intervals. The audits should show whether the ISMS:
- a) conforms to:
  1. the organization's own requirements for its ISMS, and
  2. the requirements of ISO/IEC 27001.
- b) is effectively implemented and maintained.

**9.2.2 Internal audit programme**
- The ...
-------------------------------------------------------

👤 User: How does that relate to the management review in 9.3?


🤖 Assistant: Internal audit (9.2) feeds into management review (9.3). The audit reports the findings, and top management uses them to judge how well the ISMS is working.

**The direct link**
- Clause 9.3.2 says the management review inputs must include feedback on information security performance, including trends in **audit results** (d.3).
- Clause 9.2.2 says audit results must be reported to relevant manage...
-------------------------------------------------------

👤 User: What outputs must the management review produce?


🤖 Assistant: Clause 9.3.3 (Management review results, pages 15–16) sets out what the review must produce.

**Required outputs**
- The results must include **decisions on continual improvement opportunities**.
- The results must include decisions on **any needs for changes to the ISMS**.

**Documented evidence**
- Documented information must be available as evidence of the results of management reviews (page 16...
-------------------------------------------------------


---
## 🛠️ Section 8: Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic

The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

**What the CLI does:**
1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [ ]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

In [ ]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)
# (kept on ONE line so it also works on Windows)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py --pdf_path "{PDF_PATH}" --model gpt-4o-2024-11-20 --toc-check-pages 20 --max-pages-per-node 10 --if-add-node-summary yes

In [ ]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r", encoding="utf-8") as f:
    local_tree = json.load(f)

# The local runner wraps the tree as {"doc_name": ..., "structure": [...]}
if isinstance(local_tree, dict):
    local_tree = local_tree.get("structure", local_tree.get("result", []))

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the requirements for the information security policy."
answer = vectorless_rag(query, local_tree)

---
## 📊 Section 9: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** (reported by PageIndex) | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure


In [ ]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Same query, two retrieval styles — the PageIndex half runs for real.

q = "What documented information is required for risk treatment?"

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print(f"""
query_vec = embed_model.encode("{q}")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: "risk" + "documented information" appear in 6.1.2, 6.1.3, 7.5, 8.2, 8.3
#          and the Contents page — similarity can't tell which one is the requirement
# No section context, no page ranges
""")

print("=" * 55)
print("PAGEINDEX approach (actual, live):")
print("=" * 55)
demo = llm_tree_search(q, pageindex_tree)
picked = find_nodes_by_ids(pageindex_tree, demo.get("node_list", []))
print("Node IDs :", demo.get("node_list"))
for n in picked:
    print(f"  → [{n['node_id']}] {n['title']}  (p.{n.get('start_index','?')}-{n.get('end_index','?')})")
print("\nReasoning:", demo.get("thinking", "")[:400])


---
## 🧹 Section 10: Cleanup

Delete documents from the PageIndex cloud when you're done  
to keep your storage clean.


In [25]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")

ℹ️ Deletion commented out — uncomment when you're done with this doc_id


---
## ✅ Summary

You've now built a complete **Vectorless RAG** system with PageIndex.

### What you built:

1. **`llm_tree_search()`** — LLM reasons over document tree to find relevant nodes
2. **`find_nodes_by_ids()`** — Retrieve actual section content from tree
3. **`generate_answer()`** — LLM produces cited, grounded answers
4. **`vectorless_rag()`** — Full pipeline combining all 3 steps
5. **`expert_rag()`** — Domain-guided retrieval without any fine-tuning
6. **Chat API** — Zero-setup document Q&A

### Key takeaways:

- `Similarity ≠ Relevance` — the fundamental flaw of vector search
- Tree-based reasoning gives you **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection is just **prompt engineering** — no model training needed
- 98.7% on FinanceBench vs ~80% for vector RAG

---

### 🔗 Resources
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
- Chat Platform: https://chat.pageindex.ai
- Blog: https://pageindex.ai/blog/pageindex-intro

---
*Crash course by [@krishnaik06](https://youtube.com/@krishnaik06) | [krishnaik.in/liveclasses](https://krishnaik.in/liveclasses)*
